# Automatic Speech Recognition using Whisper and OpenVINO with Generate API

[Whisper](https://openai.com/index/whisper/) is an automatic speech recognition (ASR) system trained on 680,000 hours of multilingual and multitask supervised data collected from the web.


Whisper is a Transformer based encoder-decoder model, also referred to as a sequence-to-sequence model. It maps a sequence of audio spectrogram features to a sequence of text tokens. First, the raw audio inputs are converted to a log-Mel spectrogram by action of the feature extractor. Then, the Transformer encoder encodes the spectrogram to form a sequence of encoder hidden states. Finally, the decoder autoregressively predicts text tokens, conditional on both the previous tokens and the encoder hidden states.

You can see the model architecture in the diagram below:

![whisper_architecture.svg](https://user-images.githubusercontent.com/29454499/204536571-8f6d8d77-5fbd-4c6d-8e29-14e734837860.svg)


In this tutorial, we consider how to run Whisper using OpenVINO with Generate API. We will use pre-converted models from the [OpenVINO collection on HuggingFace](https://huggingface.co/collections/OpenVINO/speech-to-text) or convert models locally using [Hugging Face Optimum Intel](https://huggingface.co/docs/optimum/intel/index). To simplify the user experience, we will use [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai) for [Whisper automatic speech recognition scenarios](https://github.com/openvinotoolkit/openvino.genai/blob/master/samples/python/whisper_speech_recognition/README.md).

The notebook supports both audio transcription and video subtitle generation.


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/whisper-asr-genai/whisper-asr-genai.ipynb" />


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model and download](#Select-model-and-download)
    - [Select model variant](#Select-model-variant)
    - [Download or convert model](#Download-or-convert-model)
- [Run inference with WhisperPipeline](#Run-inference-with-WhisperPipeline)
    - [Audio transcription](#Audio-transcription)
    - [Transcription with timestamps](#Transcription-with-timestamps)
    - [Multilingual speech translation](#Multilingual-speech-translation)
- [Video subtitle generation](#Video-subtitle-generation)
- [Interactive demo](#Interactive-demo)


## Prerequisites
[back to top ⬆️](#Table-of-contents:)


In [ ]:
import platform


%pip install -U "openvino>=2026.1" "openvino-tokenizers>=2026.1" "openvino-genai>=2026.1"
%pip install --extra-index-url https://download.pytorch.org/whl/cpu \
"git+https://github.com/huggingface/optimum-intel.git" \
"nncf>=3.0.0" \
"torch==2.11" \
"gradio>=6.0" \
"transformers" \
"datasets<4" \
"huggingface-hub>=0.26.5" \
"yt_dlp>=2024.8.6" \
"moviepy" \
"python-ffmpeg<=1.0.16" \
"librosa" \
"soundfile>=0.12" \
"typing_extensions>=4.9"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

In [ ]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("whisper-asr-genai.ipynb")

## Select model and download
[back to top ⬆️](#Table-of-contents:)

### Select model variant

You may choose from a variety of Whisper models available on [HuggingFace](https://huggingface.co/openai), including multilingual models and English-only [Distil-Whisper](https://huggingface.co/distil-whisper) variants that offer faster inference with minimal quality loss.

For your convenience, we provide a [collection of pre-converted OpenVINO models](https://huggingface.co/collections/OpenVINO/speech-to-text) on HuggingFace Hub. You can skip the model conversion step by selecting one of the available models. If you want to reproduce the conversion process locally, please unset the **Use preconverted models** checkbox.


In [ ]:
import ipywidgets as widgets

model_ids = {
    "Multilingual models": [
        "openai/whisper-large-v3-turbo",
        "openai/whisper-large-v3",
        "openai/whisper-large-v2",
        "openai/whisper-large",
        "openai/whisper-medium",
        "openai/whisper-small",
        "openai/whisper-base",
    ],
    "English-only models": [
        "distil-whisper/distil-large-v3",
        "distil-whisper/distil-large-v2",
        "distil-whisper/distil-medium.en",
        "distil-whisper/distil-small.en",
        "openai/whisper-medium.en",
        "openai/whisper-small.en",
        "openai/whisper-base.en",
        "openai/whisper-tiny.en",
    ],
}

model_type = widgets.Dropdown(
    options=model_ids.keys(),
    value="Multilingual models",
    description="Model type:",
    disabled=False,
)

model_type

In [ ]:
model_id = widgets.Dropdown(
    options=model_ids[model_type.value],
    value=model_ids[model_type.value][-1],
    description="Model:",
    disabled=False,
)

model_id

In [ ]:
precision = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="FP16",
    description="Precision:",
    disabled=False,
)

use_preconverted = widgets.Checkbox(
    value=True,
    description="Use preconverted models",
    disabled=False,
)

display(precision, use_preconverted)

### Download or convert model
[back to top ⬆️](#Table-of-contents:)

Model conversion and optimization is time- and memory-consuming process. For your convenience, we provide a [collection](https://huggingface.co/collections/OpenVINO/speech-to-text) of optimized models on HuggingFace hub. You can skip the model conversion step by selecting one of the available models. If you want to reproduce the optimization process locally, please unset **Use preconverted models** checkbox.


In [ ]:
import re


def get_ov_model_id(hf_model_id: str, prec: str = "fp16") -> str:
    """Map original HuggingFace model ID to OpenVINO pre-converted repo ID."""
    name = hf_model_id.split("/")[-1]
    org = hf_model_id.split("/")[0]
    if org == "distil-whisper":
        # distil-whisper/distil-large-v3 → distil-whisper-large-v3
        name = name.replace("distil-", "", 1)
        ov_name = f"distil-whisper-{name}"
    else:
        # openai/whisper-base → whisper-base
        ov_name = name
    return f"OpenVINO/{ov_name}-{prec.lower()}-ov"


def download_or_convert(hf_model_id: str, prec: str, use_preconv: bool) -> Path:
    """Download preconverted model or convert locally."""
    name = hf_model_id.split("/")[-1]
    name = re.sub(r'[<>:"/\\|?*]', "_", name)
    model_dir = Path(name)

    if (model_dir / "openvino_encoder_model.xml").exists():
        print(f"✅ {hf_model_id} model already available in {model_dir}")
        return model_dir

    if use_preconv:
        ov_model_hub_id = get_ov_model_id(hf_model_id, prec)
        import huggingface_hub as hf_hub

        hub_api = hf_hub.HfApi()
        if hub_api.repo_exists(ov_model_hub_id):
            print(f"⌛ Found preconverted {prec} model. Downloading {ov_model_hub_id}...")
            hf_hub.snapshot_download(ov_model_hub_id, local_dir=model_dir)
            print(f"✅ {prec} {hf_model_id} model downloaded to {model_dir}")
            return model_dir
        else:
            print(f"⚠️ Preconverted model {ov_model_hub_id} not found on HuggingFace Hub. Falling back to local conversion.")

    from cmd_helper import optimum_cli

    print(f"⌛ Converting {hf_model_id} to OpenVINO IR format...")
    optimum_cli(hf_model_id, model_dir)
    print(f"✅ {hf_model_id} model converted and can be found in {model_dir}")
    return model_dir


model_path = download_or_convert(model_id.value, precision.value, use_preconverted.value)

## Run inference with WhisperPipeline
[back to top ⬆️](#Table-of-contents:)

To simplify user experience we will use [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai/blob/master/samples/python/whisper_speech_recognition/README.md). We will create a pipeline with `WhisperPipeline`. You can construct it straight away from the folder with the converted model. It will automatically load the `model`, `tokenizer`, `detokenizer` and default `generation configuration`.


In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU")

device

In [ ]:
import openvino_genai as ov_genai

ov_pipe = ov_genai.WhisperPipeline(str(model_path), device=device.value)

### Audio transcription
[back to top ⬆️](#Table-of-contents:)

Let's start with a simple audio transcription task. We will download a sample audio file and use the Whisper pipeline to transcribe it.


In [ ]:
from notebook_utils import download_file

en_example_short = Path("data", "courtroom.wav")

if not en_example_short.exists():
    download_file(
        "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/courtroom.wav",
        en_example_short.name,
        directory=en_example_short.parent,
    )

In [ ]:
import librosa
import IPython.display as ipd

en_raw_speech, samplerate = librosa.load(str(en_example_short), sr=16000)

genai_result = ov_pipe.generate(en_raw_speech)

display(ipd.Audio(en_raw_speech, rate=samplerate))
print(f"Result: {genai_result}")

### Transcription with timestamps
[back to top ⬆️](#Table-of-contents:)

Whisper can provide phrase-level timestamps for audio. We specify `return_timestamps=True` for the `generate` method.

`generate` method with `return_timestamps` set to `True` will return `chunks`, which contain attributes: `text`, `start_ts` and `end_ts` in seconds.


In [ ]:
genai_result_timestamps = ov_pipe.generate(en_raw_speech, return_timestamps=True)

for segment in genai_result_timestamps.chunks:
    print(f"{segment.start_ts}sec. ---> {segment.end_ts}sec.")
    print(f"{segment.text}\n")

In [ ]:
import numpy as np


def print_perf_metrics(perf_metrics):
    print(f"Pipeline loading time {perf_metrics.get_load_time()} ms")
    print(f"Number of generated tokens: {perf_metrics.get_num_generated_tokens()}")
    print(f"Generate duration: {perf_metrics.get_generate_duration().mean:.2f} ms")
    print(f"TTFT: {perf_metrics.get_ttft().mean:.2f} ms")
    print(f"TPOT: {perf_metrics.get_tpot().mean:.2f} ms/token")
    print(f"Throughput: {perf_metrics.get_throughput().mean:.2f} tokens/s")
    print(f"Encoder average inference time: {perf_metrics.get_features_extraction_duration().mean:.2f} ms")
    infer_times = perf_metrics.raw_metrics.token_infer_durations
    first_token_latency = infer_times[0]
    second_token_latency = np.mean(infer_times[1:])
    print(f"Decoder first token latency {first_token_latency:.2f} ms/tokens")
    print(f"Decoder second token latency {second_token_latency:.2f} ms/tokens")


print_perf_metrics(genai_result_timestamps.perf_metrics)

### Multilingual speech translation
[back to top ⬆️](#Table-of-contents:)

If a multilingual model was chosen, let's see how the `translate` task works. The model will translate audio from the selected language into English. A complete list of languages supported by the model can be found in the [paper](https://cdn.openai.com/papers/whisper.pdf).


In [ ]:
languages = {"japanese": "ja_jp", "dutch": "da_dk", "french": "fr_fr", "spanish": "ca_es", "italian": "it_it", "portuguese": "pt_br", "polish": "pl_pl"}
languages_genai = {
    "japanese": "<|ja|>",
    "dutch": "<|da|>",
    "french": "<|fr|>",
    "spanish": "<|es|>",
    "italian": "<|it|>",
    "portuguese": "<|pt|>",
    "polish": "<|pl|>",
}

SAMPLE_LANG = None
if model_type.value == "Multilingual models":
    SAMPLE_LANG = widgets.Dropdown(
        options=languages.keys(),
        value="italian",
        description="Dataset language:",
        disabled=False,
    )

SAMPLE_LANG

In [ ]:
from datasets import load_dataset

mls_dataset = None
if model_type.value == "Multilingual models":
    mls_dataset = load_dataset("google/fleurs", languages[SAMPLE_LANG.value], split="test", streaming=True, trust_remote_code=True)
    mls_dataset = iter(mls_dataset)
    mls_example = next(mls_dataset)

In [ ]:
if model_type.value == "Multilingual models":
    sample = mls_example["audio"]

    genai_result_ml = ov_pipe.generate(sample["array"], max_new_tokens=100, task="translate", language=languages_genai[SAMPLE_LANG.value])

    display(ipd.Audio(sample["array"], rate=sample["sampling_rate"]))
    print(f"Reference: {mls_example['raw_transcription']}")
    print(f"\nResult: {genai_result_ml}")

## Video subtitle generation
[back to top ⬆️](#Table-of-contents:)

Whisper can also be used to generate subtitles for video files. The pipeline below extracts audio from a video, transcribes it using the Whisper model with timestamps, and generates an `.srt` subtitle file.

![whisper_pipeline.png](https://user-images.githubusercontent.com/29454499/204536733-1f4342f7-2328-476a-a431-cb596df69854.png)


In [ ]:
from notebook_utils import download_file

video_file = Path("downloaded_video.mp4")

if not video_file.exists():
    download_file(
        "https://storage.openvinotoolkit.org/repositories/openvino_notebooks/data/data/video/Sheldon%20Cooper%20Jim%20Parsons%20at%20Intels%20Lab.mp4",
        filename=video_file.name,
    )

In [ ]:
if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/whisper-asr-genai/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

In [ ]:
task = widgets.Select(
    options=["transcribe", "translate"],
    value="transcribe",
    description="Select task:",
    disabled=False,
)
task

In [ ]:
import math
from gradio_helper import get_audio, prepare_srt

inputs, duration = get_audio(video_file)

# Increase max_length for audio longer than 30 seconds
audio_duration_sec = len(inputs["raw"]) / 16000
if audio_duration_sec > 30:
    config = ov_pipe.get_generation_config()
    chunk_num = math.ceil(audio_duration_sec / 30)
    config.max_length = chunk_num * config.max_length
    ov_pipe.set_generation_config(config)

transcription = ov_pipe.generate(inputs["raw"], task=task.value, return_timestamps=True).chunks

srt_lines = prepare_srt(transcription, filter_duration=duration)
print("".join(srt_lines))

print("Generated subtitles:")

# Save transcription

with video_file.with_suffix(".srt").open("w") as f:
    f.writelines(srt_lines)

In [ ]:
widgets.Video.from_file(video_file, loop=False, width=800, height=800)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

We provide an interactive demo using the Gradio interface with two tabs:
- **Audio Transcription**: test the model on your own audio data (upload or record using microphone)
- **Video Subtitles**: generate `.srt` subtitle files from video input


In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe, model_id.value, video_file)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/